# Healthcare Challenge 3 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 3: Discharge Readiness Prediction**.

**Goal**: Predict `discharge_ready_day11` (0/1) for each hospital stay
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular stay data with a simple Random Forest classifier.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your-api-key-here",        # Get from your team dashboard
    team_name="your-team-name-here"     # Your exact team name
)

# Load data from PVC paths
print("📂 Loading Healthcare Challenge 3 data...")

# Load hospital stays data
train_stays = pd.read_csv("/home/jovyan/shared/datasets/Healthcare/stays_train.csv")
test_stays = pd.read_csv("/home/jovyan/shared/datasets/Healthcare/stays_test.csv")

print(f"✅ Data loaded:")
print(f"   Train stays: {train_stays.shape}")
print(f"   Test stays: {test_stays.shape}")
print(f"   Train columns: {list(train_stays.columns)}")
print(f"   Test columns: {list(test_stays.columns)}")


In [ ]:
# 2. Tabular-Only Baseline Model and Predictions

# From data inspection - stays columns:
# stay_id, patient_id, unit_type, admission_reason, discharge_ready_day11 (train only)

# For this simple baseline, we'll use basic encoding of categorical features
# Convert unit_type and admission_reason to numeric codes
unit_encoder = LabelEncoder()
reason_encoder = LabelEncoder()

# Fit encoders on training data
unit_encoded_train = unit_encoder.fit_transform(train_stays['unit_type'])
reason_encoded_train = reason_encoder.fit_transform(train_stays['admission_reason'])

# Transform test data
unit_encoded_test = unit_encoder.transform(test_stays['unit_type'])
reason_encoded_test = reason_encoder.transform(test_stays['admission_reason'])

print(f"📊 Using encoded categorical features:")
print(f"   Unit types: {list(unit_encoder.classes_)}")
print(f"   Admission reasons: {list(reason_encoder.classes_)}")

# Prepare training data
X_train = pd.DataFrame({
    'patient_id': train_stays['patient_id'],
    'unit_type_encoded': unit_encoded_train,
    'admission_reason_encoded': reason_encoded_train
})
y_train = train_stays['discharge_ready_day11']  # Binary target (0/1)

# Prepare test data
X_test = pd.DataFrame({
    'patient_id': test_stays['patient_id'],
    'unit_type_encoded': unit_encoded_test,
    'admission_reason_encoded': reason_encoded_test
})

# Train simple Random Forest baseline
print("🤖 Training Random Forest classifier...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Create submission file (format: stay_id,discharge_ready_day11)
submission_df = pd.DataFrame({
    'stay_id': test_stays['stay_id'],
    'discharge_ready_day11': predictions
})

# Save predictions
submission_df.to_csv("healthcare_challenge3_predictions.csv", index=False)
print(f"✅ Predictions saved: {submission_df.shape[0]} predictions")
print(f"   Preview: {submission_df.head(3)}")
print(f"   Discharge ready rate: {predictions.mean():.3f} ({predictions.sum()} ready out of {len(predictions)})")


In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 3, "healthcare_challenge3_predictions.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. You've completed all Healthcare challenges!")
